# Knowledge Graph Construction & Risk Clustering

Build NetworkX knowledge graph from submissions, extract risk clusters, and identify control impact patterns.

**Objectives:**
1. Load sample deltas and PDF fields
2. Construct KG with customer, submission, control, broker nodes
3. Run KMeans clustering to identify risk groups
4. Compute control impact scores (which controls predict approval?)
5. Export results for dashboard visualization

In [ ]:
import os
from pathlib import Path
import pandas as pd
import networkx as nx
from sklearn.cluster import KMeans
import numpy as np

os.chdir('..')

print(f'Working directory: {Path.cwd()}')

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)

print('[SETUP] Libraries loaded')

## 1. Load Data

In [ ]:
pdf_extracted = pd.read_csv('data/parsed/pdf_extracted_fields.csv')
sample_deltas = pd.read_csv('data/parsed/sample_deltas.csv')
recommendations = pd.read_csv('data/parsed/sample_recommendations.csv')
all_submissions = pd.read_csv('data/parsed/all_submissions.csv')

print(f'Loaded PDF extracted records: {len(pdf_extracted)}')
print(f'Loaded sample deltas: {len(sample_deltas)}')
print(f'Loaded sample recommendations: {len(recommendations)}')
print(f'Loaded Excel submissions: {len(all_submissions)}')

print('\nPDF columns:', list(pdf_extracted.columns))
print('Delta columns:', list(sample_deltas.columns))

## 2. Build Knowledge Graph

In [ ]:
G = nx.Graph()

# Add company nodes
for company in pdf_extracted['company_name'].unique():
    G.add_node(company, node_type='company')

# Add PDF submission nodes and company relationships
for _, row in pdf_extracted.iterrows():
    pdf_node = f"pdf::{row['company_name']}::{row['pdf_file']}"
    G.add_node(pdf_node, node_type='pdf', revenue=row['revenue_millions'], employees=row['employee_count'])
    G.add_edge(row['company_name'], pdf_node, relationship='submitted')

# Add control nodes and edges
control_cols = [c for c in pdf_extracted.columns if c.startswith('control_')]
for control in control_cols:
    G.add_node(control, node_type='control')

for _, row in pdf_extracted.iterrows():
    pdf_node = f"pdf::{row['company_name']}::{row['pdf_file']}"
    for control in control_cols:
        if row.get(control, False):
            G.add_edge(pdf_node, control, relationship='has_control')

print(f'Graph nodes: {G.number_of_nodes()}')
print(f'Graph edges: {G.number_of_edges()}')
print('Sample company node:', list(G.nodes(data=True))[:5])

## 3. Cluster Companies by Risk Profile

In [ ]:
company_features = []
company_ids = []
for company, group in pdf_extracted.groupby('company_name'):
    controls_sum = group[control_cols].sum().to_dict()
    revenue_mean = group['revenue_millions'].mean() if not group['revenue_millions'].isna().all() else 0
    employees_mean = group['employee_count'].mean() if not group['employee_count'].isna().all() else 0
    feature_vector = [revenue_mean, employees_mean] + [controls_sum[c] for c in control_cols]
    company_features.append(feature_vector)
    company_ids.append(company)

company_features = np.nan_to_num(np.array(company_features), nan=0.0)

kmeans = KMeans(n_clusters=min(4, len(company_ids)), random_state=42)
clusters = kmeans.fit_predict(company_features)

cluster_df = pd.DataFrame({'company_name': company_ids, 'cluster': clusters})
print(cluster_df.value_counts('cluster'))

# Attach cluster labels to nodes
for company, cluster in zip(company_ids, clusters):
    if G.has_node(company):
        G.nodes[company]['cluster'] = int(cluster)

print('\nCluster assignments:')
print(cluster_df.head())

## 4. Compute Control Impact Scores

In [ ]:
control_impact = []
for control in control_cols:
    has_control = pdf_extracted[pdf_extracted[control] == True]
    no_control = pdf_extracted[pdf_extracted[control] == False]
    
    if len(has_control) == 0 or len(no_control) == 0:
        continue
    
    score = has_control['revenue_millions'].mean() - no_control['revenue_millions'].mean()
    control_impact.append({'control': control, 'sample_size': len(has_control), 'revenue_diff': score})

impact_df = pd.DataFrame(control_impact).sort_values('revenue_diff', ascending=False)
print('Top control impact by revenue difference:')
print(impact_df.head(10).to_string(index=False))

## 5. Export Graph & Cluster Results

In [ ]:
nx.write_graphml(G, 'data/parsed/knowledge_graph.graphml')
cluster_df.to_csv('data/parsed/company_clusters.csv', index=False)
impact_df.to_csv('data/parsed/control_impact.csv', index=False)

print('Saved graph to data/parsed/knowledge_graph.graphml')
print('Saved company cluster assignments to data/parsed/company_clusters.csv')
print('Saved control impact summary to data/parsed/control_impact.csv')

print('\n✅ Knowledge graph construction notebook ready')